# Train Attention Evidence Aggregator on Google Colab T4

Notebook wrapper for ``scripts/train_evidence_aggregator.py``. Every
real piece of logic lives in the script — this notebook just provides
the GPU runtime environment.

**Prerequisites**: a GitHub Personal Access Token (or use HTTPS with
a public clone) — pass to ``git clone`` via ``$GITHUB_TOKEN`` when
the repo is private.


## Cell 1 — Clone the repository

In [ ]:
import os
REPO_URL = "https://github.com/HQuan1428/Stock-trend-forecasting-system.git"
!git clone "$REPO_URL" Stock-trend-forecasting-system
!git checkout Update-Dataset
!cd Stock-trend-forecasting-system

## Cell 2 — Install dependencies (PyTorch CUDA + Hugging Face)

In [ ]:
!pip install -r requirements.txt

## Cell 3 — Confirm GPU T4 is reachable

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
!nvidia-smi

## Cell 4 — Train the Attention Evidence Aggregator

In [ ]:
from scripts.train_evidence_aggregator import (
    build_training_data,
    train_model,
    evaluate_model,
    split_samples,
)
import torch

samples = build_training_data("data/real_dataset.csv")
print(f"Total samples: {len(samples)}")
data = split_samples(samples)
print(f"train={len(data.train)} val={len(data.val)} test={len(data.test)}")

model, history = train_model(
    data,
    device="cuda" if torch.cuda.is_available() else "cpu",
    epochs=50,
    lr=1e-3,
    batch_size=8,
    patience=5,
)

## Cell 5 — Evaluate the best checkpoint on the held-out test split

In [ ]:
metrics = evaluate_model(
    model,
    data.test,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Macro-F1: {metrics['macro_f1']:.4f}")
print(f"Confusion matrix (rows=pred, cols=actual [UP, DOWN, HOLD]):")
for row in metrics["confusion"]:
    print(" ", row)
print(f"N test samples: {metrics['n_samples']}")

## Cell 6 — Benchmark eager vs `torch.compile` (Inductor / Triton)

Per the assignment: at least one cell demonstrating that
`torch.compile` produces a working CUDA kernel on T4.

In [ ]:
from scripts.train_evidence_aggregator import benchmark_compile

bench = benchmark_compile(
    model,
    data.test,
    device="cuda" if torch.cuda.is_available() else "cpu",
    warm_up=5,
    measure_passes=50,
)
print(bench)

## Cell 7 — Save checkpoint to disk, then to Google Drive

After this cell runs, commit the resulting
`models/evidence_aggregator_v1.pt` to the repository:

```bash
cd Stock-trend-forecasting-system
git add models/evidence_aggregator_v1.pt
git commit -m "Update Attention checkpoint after Colab T4 training"
git push origin Update-Dataset
```

In [ ]:
import os
import torch
from pathlib import Path

ckpt_path = Path("models/evidence_aggregator_v1.pt")
ckpt_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), ckpt_path)
print(f"Saved: {ckpt_path} ({os.path.getsize(ckpt_path)} bytes)")

# Optional: copy to Google Drive if mounted.
drive_dir = Path("/content/drive/MyDrive/Stock-trend-forecasting-system-models")
try:
    drive_dir.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy(ckpt_path, drive_dir / ckpt_path.name)
    print(f"Copied checkpoint to: {drive_dir / ckpt_path.name}")
except Exception as exc:
    print(f"Drive copy skipped: {exc}")